In [1]:
!python -m pip install -q -U google-adk litellm openai requests python-dotenv pandas

print("✅ Dependencies installed.")

✅ Dependencies installed.


# 2.读取 NVIDIA API Key（优先从 Colab Secrets 读取）

In [2]:
import os
from getpass import getpass

def setup_nvidia_api():
    api_key = None

    try:
        from google.colab import userdata
        api_key = userdata.get("NVIDIA_NIM_API_KEY")
        if api_key:
            print("✅ 已从 Colab Secrets 读取 NVIDIA_NIM_API_KEY")
    except Exception:
        pass

    if not api_key:
        api_key = getpass("请输入你的 NVIDIA_NIM_API_KEY：").strip()

    if not api_key:
        raise ValueError("没有读取到 NVIDIA_NIM_API_KEY，无法继续。")

    os.environ["NVIDIA_NIM_API_KEY"] = api_key
    os.environ.setdefault("NVIDIA_NIM_API_BASE", "https://integrate.api.nvidia.com/v1")

    print("✅ NVIDIA API 环境变量配置完成。")
    print("NVIDIA_NIM_API_BASE =", os.environ["NVIDIA_NIM_API_BASE"])

setup_nvidia_api()

✅ NVIDIA API 环境变量配置完成。
NVIDIA_NIM_API_BASE = https://integrate.api.nvidia.com/v1


# 3.导入ADK组件

In [3]:
from typing import Any, Dict

from google.adk.agents import Agent, LlmAgent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import DatabaseSessionService, InMemorySessionService
from google.adk.runners import Runner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

print("✅ ADK 组件导入成功。")

/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey
/root/.pyenv/versions/3.11.1/lib/python3.11/site-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


✅ ADK 组件导入成功。


# 4.全局配置 + run_session() 辅助函数

In [4]:
# 全局常量
APP_NAME = "default"
USER_ID = "default"
SESSION = "default"
MODEL_NAME = "nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5"

# 封装模型创建函数
def build_model():
    return LiteLlm(model=MODEL_NAME)

# 异步会话运行函数
async def run_session(
    runner_instance: Runner,
    user_queries: list[str] | str = None,
    session_name: str = "default",
):
    print(f"\n### Session: {session_name}") #打印当前 session，看清楚当前是哪一个 session。

    app_name = runner_instance.app_name #取出 app_name

    try:
        session = await session_service.create_session(
            app_name=app_name,
            user_id=USER_ID,
            session_id=session_name,
        )
        print("✅ 新 session 已创建。")
    except Exception:
        session = await session_service.get_session(
            app_name=app_name,
            user_id=USER_ID,
            session_id=session_name,
        )
        print("✅ 已读取已有 session。")

    if not user_queries:
        print("⚠️ 没有传入查询内容。")
        return

    # 单条字符串转成列表
    if isinstance(user_queries, str):
        user_queries = [user_queries]

    # 遍历每一条用户消息
    for query in user_queries:
        print(f"\nUser > {query}")
        # 构造 ADK 消息对象。ADK 不是直接接收裸字符串，而是接收结构化消息对象。
        query_content = types.Content(
            role="user",#用户说的话
            parts=[types.Part(text=query)],#消息内容列表
        )
        # 调用 runner 的 run_async 方法，传入用户 ID、会话 ID 和消息内容。
        async for event in runner_instance.run_async(
            user_id=USER_ID,
            session_id=session.id,
            new_message=query_content,
        ):
            # 只打印文本输出
            if event.content and event.content.parts:
                text = event.content.parts[0].text
                if text and text != "None":
                    print(f"{MODEL_NAME} > {text}")

print("✅ 全局配置与辅助函数定义完成。")

✅ 全局配置与辅助函数定义完成。


# 5.初始化最基础的 Stateful Agent

In [5]:
root_agent = Agent(
    model=build_model(),
    name="text_chat_bot",
    description="A text chatbot",
)

session_service = InMemorySessionService() #创建内存型 session service。session 只存在当前 Python 进程内存里。

runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

print("✅ Stateful agent 初始化完成。")
print("APP_NAME =", APP_NAME)
print("USER_ID =", USER_ID)
print("MODEL_NAME =", MODEL_NAME)
print("Session Service =", session_service.__class__.__name__)

✅ Stateful agent 初始化完成。
APP_NAME = default
USER_ID = default
MODEL_NAME = nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5
Session Service = InMemorySessionService


# 6.测试同一个 session 是否记住上下文

In [7]:
# 第一个参数：runner，第二个参数：两轮用户消息，第三个参数：session 名
await run_session(
    runner,
    [
        "Hi, I am Sam! What is the capital of United States?",
        "Hello! What is my name?",
    ],
    "stateful-agentic-session",
)


### Session: stateful-agentic-session
✅ 新 session 已创建。

User > Hi, I am Sam! What is the capital of United States?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 
Okay, the user is Sam, and they asked for the capital of the United States. Let me start by recalling the correct answer. The capital of the U.S. is Washington, D.C. I should make sure to mention that D.C. stands for District of Columbia. But wait, sometimes people might confuse it with other cities like New York or Philadelphia, which were former capitals. Should I include that info? The user might not know that Washington, D.C. is a district, not a state. Also, mentioning that it's home to important landmarks like the Capitol and the White House could be helpful. Let me check if there's any recent change, but I don't think so. The answer should be straightforward. Let me structure it clearly: answer the question directly, then add a bit more context to be informative without overwhelming the user. Alright, that shou

# 7.观察 InMemorySessionService 的局限

In [6]:
# 这个单元建议在“重启运行时之后”再执行，才能明显观察 InMemorySessionService 的局限性
await run_session(
    runner,
    [
        "What did I ask you about earlier?",
        "And remind me, what's my name?",
    ],
    "stateful-agentic-session",
)


### Session: stateful-agentic-session
✅ 新 session 已创建。

User > What did I ask you about earlier?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 
Okay, the user is asking, "What did I ask you about earlier?" Let me check the conversation history to see what they previously asked.

Looking back, the user initially provided instructions about my identity as a text chatbot. They mentioned that my internal name is "text_chat_bot" and that I'm a text chatbot. The first question they asked was about what they had asked me earlier. Since that was the first interaction, there wasn't any prior content.

Now, they're asking the same question again. But wait, in the current conversation, their last message was asking, "What did I ask you about earlier?" and my response was that there was no prior message. Now they're repeating the same question. However, looking at the history, there's no prior exchange beyond the system message and the first user query. 

Wait, the system message set up m

# 8.切换到数据库型 Session

In [ ]:
!python -m pip install aiosqlite

#创建新的聊天 agent
chatbot_agent = LlmAgent(
    model=build_model(),
    name="text_chat_bot",
    description="A text chatbot with persistent memory",
)

# 定义数据库 URL
db_url = "sqlite:///my_agent_data.db"
# 创建数据库型 session service
session_service = DatabaseSessionService(
    db_url="sqlite+aiosqlite:///my_agent_data.db"   # ← 注意 +aiosqlite 前缀
)

runner = Runner(
    agent=chatbot_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

print("✅ 已切换到 DatabaseSessionService。")
print("数据库文件：my_agent_data.db")

Looking in indexes: https://mirrors.tencent.com/pypi/simple/
✅ 已切换到 DatabaseSessionService。
数据库文件：my_agent_data.db


# 9.写入第一个数据库 session

In [10]:
await run_session(
    runner,
    [
        "Hi, I am Sam! What is the capital of the United States?",
        "Hello! What is my name?",
    ],
    "test-db-session-01",
)


### Session: test-db-session-01
✅ 新 session 已创建。

User > Hi, I am Sam! What is the capital of the United States?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 
Okay, the user is Sam, and he asked for the capital of the United States. Let me start by recalling the answer. The capital is Washington, D.C. But I should make sure that's correct. Washington, D.C., stands for District of Columbia. It's where the government is based, right? The White House, Capitol Building, etc.

Wait, sometimes people confuse it with other cities like New York or Philadelphia because they were capitals before. But since 1800, Washington, D.C., has been the capital. So I should mention that it's been the capital since then and maybe a bit about its history.

Also, the user might be looking for a simple answer, but adding a bit more context could be helpful. Let me check if there's any common misconception. For example, some might think it's a state, but it's a federal district. But maybe that's too d

In [11]:
await run_session(
    runner,
    [
        "What is the capital of India?",
        "Hello! What is my name?",
    ],
    "test-db-session-01",
)
#数据库里之前那轮对话确实被读回来了。


### Session: test-db-session-01
✅ 已读取已有 session。

User > What is the capital of India?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 

Okay, the user asked, "What is the capital of India?" Let me start by recalling what I know about this. The capital of India is New Delhi. I'm pretty confident about that. But wait, I should make sure there's no confusion with Old Delhi. Yes, New Delhi is the official capital, established during the British colonial period. It's where important government institutions are located, like the Parliament of India and the Rashtrapati Bhavan, the President's residence.

Now, considering the user's history, they previously asked about the capital of the U.S., and I provided a detailed answer about Washington, D.C. They also mentioned their name is Sam. Since the user seems interested in geography or capitals, maybe they're studying or just curious. I should structure the answer similarly to the previous one for consistency. 

I should mention the tran

# 10.换一个新的数据库 session
模型大概率会说类似：你还没有告诉我你的名字，我不知道你的名字
重点是：不同 session 之间默认隔离。

In [12]:
await run_session(
    runner,
    [
        "Hello! What is my name?",
    ],
    "test-db-session-02",
)


### Session: test-db-session-02
✅ 新 session 已创建。

User > Hello! What is my name?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 
Okay, the user is asking, "Hello! What is my name?" Let me think.

First, I need to recall if I have any information about the user's name. Since I'm a text chatbot with persistent memory, I should check my memory for any previous interactions where the user might have provided their name. However, if this is the first interaction, I wouldn't know their name. 

In the current conversation history, the user hasn't mentioned their name yet. The initial prompt only states that my internal name is "text_chat_bot" and my description is "A text chatbot with persistent memory." There's no information about the user's name here.

Since I don't have access to any prior interactions beyond this conversation and there's no data provided, I can't retrieve the user's name. My response should inform them that I don't know their name and ask them to provide it. That

# 11.直接查看 SQLite 数据库内容
这格不是让 agent 聊天，而是直接打开 SQLite 数据库，看看 events 表里面到底存了什么。

In [21]:
import sqlite3

# 第一步：先看看表有哪些列
with sqlite3.connect("my_agent_data.db") as conn:
    cursor = conn.cursor()
    
    # 查询 events 表的列定义
    print("=== events 表结构 ===")
    cursor.execute("PRAGMA table_info(events)")
    for col in cursor.fetchall():
        print(f"  列名: {col[1]}, 类型: {col[2]}")
    
    print("\n=== 数据预览 (前5条) ===")
    # 先查列名
    cursor.execute("SELECT * FROM events LIMIT 5")
    columns = [desc[0] for desc in cursor.description]
    print("  列:", columns)
    print()
    for row in cursor.fetchall():
        print(row)


=== events 表结构 ===
  列名: id, 类型: VARCHAR(128)
  列名: app_name, 类型: VARCHAR(128)
  列名: user_id, 类型: VARCHAR(128)
  列名: session_id, 类型: VARCHAR(128)
  列名: invocation_id, 类型: VARCHAR(256)
  列名: timestamp, 类型: DATETIME
  列名: event_data, 类型: TEXT

=== 数据预览 (前5条) ===
  列: ['id', 'app_name', 'user_id', 'session_id', 'invocation_id', 'timestamp', 'event_data']

('0be54479-7816-4fec-897c-b44e39ca8ddf', 'default', 'default', 'test-db-session-01', 'e-4438a6b2-74ba-48d6-89cf-d4a285ca4633', '2026-04-24 05:36:49.590428', '{"content": {"parts": [{"text": "Hi, I am Sam! What is the capital of the United States?"}], "role": "user"}, "invocation_id": "e-4438a6b2-74ba-48d6-89cf-d4a285ca4633", "author": "user", "actions": {"state_delta": {}, "artifact_delta": {}, "requested_auth_configs": {}, "requested_tool_confirmations": {}}, "id": "0be54479-7816-4fec-897c-b44e39ca8ddf", "timestamp": 1777009009.590428}')
('79a1f274-a62b-41e4-8e6e-dc03ba9903fc', 'default', 'default', 'test-db-session-01', 'e-4438a6b2-74b

In [24]:
import sqlite3
import json

with sqlite3.connect("my_agent_data.db") as conn:
    cursor = conn.cursor()
    
    # ✅ 用正确的列名查询
    cursor.execute("""
        SELECT session_id, timestamp, event_data 
        FROM events 
        ORDER BY timestamp
    """)
    
    rows = cursor.fetchall()
    print(f"共 {len(rows)} 条事件记录")
    print("=" * 80)
    
    for i, (session_id, ts, event_data) in enumerate(rows, 1):
        print(f"📦 记录 #{i}")
        print(f"   Session: {session_id}")
        print(f"   时间:    {ts}")
        
        # event_data 是 JSON 字符串，解析后可读性更好
        try:
            parsed = json.loads(event_data)
            # 只打印关键信息，避免刷屏
            if isinstance(parsed, dict):
                role = parsed.get("role", parsed.get("author", "N/A"))
                content = parsed.get("content", parsed.get("text", ""))
                # 如果内容太长就截断
                if len(str(content)) > 200:
                    content = str(content)[:200] + "..."
                print(f"   数据:    role={role}, content={content}")
            else:
                print(f"   原始数据: {str(parsed)[:200]}...")
        except (json.JSONDecodeError, TypeError):
            # 如果不是 JSON 就直接打印原始文本
            raw = str(event_data)
            print(f"   原始:    {raw[:200]}...")

print("✅ 查询完成!")


共 10 条事件记录
📦 记录 #1
   Session: test-db-session-01
   时间:    2026-04-24 05:36:49.590428
   数据:    role=user, content={'parts': [{'text': 'Hi, I am Sam! What is the capital of the United States?'}], 'role': 'user'}
📦 记录 #2
   Session: test-db-session-01
   时间:    2026-04-24 05:36:49.623366
   数据:    role=text_chat_bot, content={'parts': [{'text': '\nOkay, the user is Sam, and he asked for the capital of the United States. Let me start by recalling the answer. The capital is Washington, D.C. But I should make sure that\'s co...
📦 记录 #3
   Session: test-db-session-01
   时间:    2026-04-24 05:37:04.311582
   数据:    role=user, content={'parts': [{'text': 'Hello! What is my name?'}], 'role': 'user'}
📦 记录 #4
   Session: test-db-session-01
   时间:    2026-04-24 05:37:04.340642
   数据:    role=text_chat_bot, content={'parts': [{'text': '\nOkay, let\'s see. The user just asked, "Hello! What is my name?" Hmm. I need to figure out their name based on the conversation history.\n\nLooking back at the pr

# 12.Events Compaction（事件压缩）
核心思想是：
对话轮数越来越多；
event 越来越长；
上下文成本越来越高；
于是系统可以把旧 event 压缩成摘要。

## 12.1创建带 Compaction 的 App
把“事件压缩”能力加到应用里。

In [26]:
# 创建一个启用了事件压缩的应用
research_app_compacting = App(
    name="research_app_compacting",
    root_agent=chatbot_agent,
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=3, # 大约每累计一定间隔触发一次压缩
        overlap_size=1, # 重叠大小，即在压缩过程中保留的旧事件数量
    ),
)

# 重新创建数据库 session service
db_url = "sqlite+aiosqlite:///my_agent_data.db"   # 变量本身就要带前缀
session_service = DatabaseSessionService(db_url=db_url)


research_runner_compacting = Runner(
    app=research_app_compacting,
    session_service=session_service,
)

print("✅ 已启用 Events Compaction。")

✅ 已启用 Events Compaction。


/tmp/ipykernel_257074/2806585111.py:5: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


## 12.2多轮对话，触发 Compaction


In [27]:
await run_session(
    research_runner_compacting,
    "What is the latest news about AI in healthcare?",
    "compaction_demo",
)

await run_session(
    research_runner_compacting,
    "Are there any new developments in drug discovery?",
    "compaction_demo",
)

await run_session(
    research_runner_compacting,
    "Tell me more about the second development you found.",
    "compaction_demo",
)

await run_session(
    research_runner_compacting,
    "Who are the main companies involved in that?",
    "compaction_demo",
)


### Session: compaction_demo
✅ 新 session 已创建。

User > What is the latest news about AI in healthcare?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 
Okay, the user is asking about the latest news in AI in healthcare. Let me start by recalling what I know. I need to make sure the information is up-to-date, but since my knowledge cutoff is July 2024, I should mention that first.

First, I should break down the answer into different areas where AI is impacting healthcare. Maybe start with diagnostics and imaging. I remember reading that AI is being used for early detection of diseases like cancer. For example, there's been progress in breast cancer detection using mammography analysis. There might be new studies showing improved accuracy over traditional methods.

Then there's personalized medicine. AI can analyze genetic data and patient records to tailor treatments. Maybe there are updates on how machine learning models are being used for drug discovery. Some companies might ha

# 12.2 检查是否真的产生了 Compaction Event

In [28]:
final_session = await session_service.get_session(
    app_name=research_runner_compacting.app_name,
    user_id=USER_ID,
    session_id="compaction_demo",
)

print("--- Searching for Compaction Summary Event ---")
found_summary = False

for event in final_session.events:
    if event.actions and event.actions.compaction:
        print("\n✅ 成功找到 Compaction Event：")
        print("Author:", event.author)
        print("\nCompacted Event 内容：")
        print(event)
        found_summary = True
        break

if not found_summary:
    print("❌ 没找到 compaction 事件。你可以再多跑几轮对话。")

--- Searching for Compaction Summary Event ---

✅ 成功找到 Compaction Event：
Author: user

Compacted Event 内容：
model_version=None content=None grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None live_session_id=None go_away=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None invocation_id='a4317fd0-04b0-492b-b15a-5b8e97052c83' author='user' actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=None, escalate=None, requested_auth_configs={}, requested_tool_confirmations={}, compaction=EventCompaction(start_timestamp=1777010032.832372, end_timestamp=1777010181.79829, compacted_content=Content(
  parts=[
    Part(
      text="""
Okay, let's see. The user asked for a summary of the conversation h

# 12.3 定义两个工具，读写用户信息

In [29]:
USER_NAME_SCOPE_LEVELS = ("temp", "user", "app")

# 把用户名字和国家写进 session state
def save_userinfo(
    tool_context: ToolContext,
    user_name: str,
    country: str,
) -> Dict[str, Any]:
    tool_context.state["user:name"] = user_name
    tool_context.state["user:country"] = country
    return {"status": "success"}

#从 session state 读出用户名字和国家
def retrieve_userinfo(tool_context: ToolContext) -> Dict[str, Any]:
    user_name = tool_context.state.get("user:name", "Username not found")
    country = tool_context.state.get("user:country", "Country not found")
    return {
        "status": "success",
        "user_name": user_name,
        "country": country,
    }

print("✅ Session state 工具创建完成。")

✅ Session state 工具创建完成。


# 12.4 创建带工具的 Agent


In [30]:
# 重置 APP_NAME 和 USER_ID
APP_NAME = "default"
USER_ID = "default"

# 创建带工具的 LlmAgent
root_agent = LlmAgent(
    model=build_model(),
    name="text_chat_bot",
    description="""A text chatbot.
Tools for managing user context:
* To record username and country when provided use `save_userinfo`.
* To fetch username and country when required use `retrieve_userinfo`.
""",
    tools=[save_userinfo, retrieve_userinfo],
)

# 创建 InMemorySessionService 实例。这一部分的重点是演示 session.state 的存在方式，
# 不一定非要落数据库。
session_service = InMemorySessionService()

runner = Runner(
    agent=root_agent,
    session_service=session_service,
    app_name=APP_NAME,
)

print("✅ 带 session state 工具的 Agent 初始化完成。")

✅ 带 session state 工具的 Agent 初始化完成。


# 12.5 测试工具是否会把用户信息写入 Session State
Agent 不只是从普通对话里“猜到”你的信息，而是已经通过工具把信息写入结构化 state。

In [31]:
await run_session(
    runner,
    [
        "Hi there, how are you doing today? What is my name?",
        "My name is Sam. I'm from Poland.",
        "What is my name? Which country am I from?",
    ],
    "state-demo-session",
)


### Session: state-demo-session
✅ 新 session 已创建。

User > Hi there, how are you doing today? What is my name?
nvidia_nim/nvidia/llama-3.3-nemotron-super-49b-v1.5 > 
Okay, the user just asked, "Hi there, how are you doing today? What is my name?" Let me break this down.

First, I need to respond to their greeting. They asked how I'm doing, so a friendly reply like "I'm doing well, thank you!" makes sense. Then, they asked, "What is my name?" That's where the tools come into play. 

Looking at the available tools, there's retrieve_userinfo which fetches username and country. But wait, have I saved any user info before? The user hasn't provided their name yet, so the system might not have it stored. If I call retrieve_userinfo now, it might return empty or default values. The description says to use save_userinfo when the user provides the name and country. Since the user hasn't given their name yet, I can't retrieve it. 

So, the correct approach is to inform the user that I don't have t

## 12.6直接检查 Session State
用户资料已经从“对话文本”升级成了“结构化状态”。

In [33]:
session = await session_service.get_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id="state-demo-session",
)

print("Session State Contents:")
print(session.state)
print("\n🔍 这里你应该能看到 user:name 和 user:country。")

Session State Contents:
{'user:name': 'Sam', 'user:country': 'Poland'}

🔍 这里你应该能看到 user:name 和 user:country。
